<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 1. Aspectos generales — Modelo 1: CLV (Customer Lifetime Value)

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Responsables: **Anyel (Lead)** y **Gianpool (Validación canal/edad y consolidación)**.

Calcula el **Customer Lifetime Value (CLV)** histórico y proyectado a 12 meses 100 % en **Apache Spark nativo** (sin transformaciones en Pandas) leyendo exclusivamente desde `../silver/merge_pyspark`.

Flujo: `../silver/merge_pyspark` ---> agregación por cliente ---> CLV histórico & proyectado 12m ---> validación canal/edad ---> `../gold/hym_clv`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder
    .appName("HYM_Lab13_01_CLV")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "50")
    .config("spark.memory.fraction", "0.8")
    .getOrCreate()
)

spark.sparkContext.setCheckpointDir("/tmp/spark_checkpoint")
spark

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 2. Rutas de capas

In [ ]:
SILVER = "../silver"
GOLD   = "../gold"

import os
os.makedirs(GOLD, exist_ok=True)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 3. Carga desde capa Silver

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Lee `../silver/merge_pyspark` conservando todos los clientes válidos (`price IS NOT NULL AND price > 0`), incluyendo compradores de 1 sola transacción para no sesgar el CLV total.

In [ ]:
%%time
df_silver = spark.read.parquet(f"{SILVER}/merge_pyspark")
df_clean = (
    df_silver
    .select("customer_id", "Fecha", "price", "sales_channel_id", "age")
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
)
df_clean.createOrReplaceTempView("clean_transactions")
print(f"Transacciones válidas en Silver: {df_clean.count():,}")

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 4. Cálculo de CLV histórico y proyectado en Spark SQL

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Métricas por cliente calculadas en una pasada SQL:

- `total_compras`, `gasto_total` (**CLV histórico**), `aov` (ticket promedio = `gasto_total / total_compras`).
- `vida_dias` (`DATEDIFF(ultima_compra, primera_compra) + 1`) y `compras_anuales` (escalado con piso mínimo de 30 días para estabilizar clientes nuevos).
- `clv_proyectado_12m = aov * compras_anuales`.

In [ ]:
%%time
df_clv_raw = spark.sql("""
WITH stats AS (
    SELECT
        customer_id,
        COUNT(1) AS total_compras,
        SUM(price) AS clv_historico,
        AVG(price) AS aov,
        MIN(Fecha) AS primera_compra,
        MAX(Fecha) AS ultima_compra,
        DATEDIFF(MAX(Fecha), MIN(Fecha)) + 1 AS vida_dias,
        MAX(age) AS age,
        ROUND(AVG(CASE WHEN sales_channel_id = 2 THEN 1.0 ELSE 0.0 END) * 100.0, 1) AS pct_online
    FROM clean_transactions
    GROUP BY customer_id
)
SELECT
    customer_id,
    CAST(total_compras AS INT) AS total_compras,
    ROUND(clv_historico, 4) AS clv_historico,
    ROUND(aov, 4) AS aov,
    primera_compra,
    ultima_compra,
    vida_dias,
    CAST(age AS INT) AS age,
    pct_online,
    ROUND(total_compras * 365.0 / GREATEST(vida_dias, 30), 2) AS freq_anual_est,
    ROUND(aov * (total_compras * 365.0 / GREATEST(vida_dias, 30)), 4) AS clv_proyectado_12m
FROM stats
""")
df_clv_raw = df_clv_raw.checkpoint()
print(f"Clientes evaluados en CLV: {df_clv_raw.count():,}")
df_clv_raw.select("total_compras", "clv_historico", "aov", "vida_dias", "clv_proyectado_12m").describe().show()

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 5. Segmentación por cuartiles CLV (percent_rank) y validación por edad/canal

<div style="background-color:#9B9B9B; color:white; padding:10px; border-radius:5px; font-size:13px">

Segmenta en 4 niveles (`CLV_Bronce`, `CLV_Plata`, `CLV_Oro`, `CLV_Platino`) usando ventanas nativas de Spark y ejecuta la validación cruzada por rango etario y canal preferido (co-responsabilidad de Gianpool).

In [ ]:
w_clv = Window.orderBy(F.col("clv_proyectado_12m").asc())
df_clv = (
    df_clv_raw
    .withColumn("clv_pct", F.percent_rank().over(w_clv))
    .withColumn(
        "segmento_clv",
        F.when(F.col("clv_pct") >= 0.75, "CLV_Platino")
         .when(F.col("clv_pct") >= 0.50, "CLV_Oro")
         .when(F.col("clv_pct") >= 0.25, "CLV_Plata")
         .otherwise("CLV_Bronce"),
    )
    .withColumn(
        "grupo_edad",
        F.when(F.col("age").isNull() | (F.col("age") <= 0), "Sin Dato")
         .when(F.col("age") < 25, "18-24")
         .when(F.col("age") < 35, "25-34")
         .when(F.col("age") < 50, "35-49")
         .otherwise("50+"),
    )
    .withColumn(
        "canal_dominante",
        F.when(F.col("pct_online") >= 50.0, "Online (Canal 2)").otherwise("Tienda Física (Canal 1)"),
    )
)

resumen_seg = (
    df_clv.groupBy("segmento_clv")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("clv_historico"), 4).alias("clv_hist_prom"),
        F.round(F.avg("clv_proyectado_12m"), 4).alias("clv_12m_prom"),
        F.round(F.avg("aov"), 4).alias("aov_prom"),
    )
    .orderBy(F.col("clv_12m_prom").desc())
)
resumen_seg.show(truncate=False)

# Validación cruzada por edad y canal (solo resumen agregado al driver)
df_val = (
    df_clv.groupBy("grupo_edad", "canal_dominante")
    .agg(
        F.count("*").alias("clientes"),
        F.round(F.avg("clv_proyectado_12m"), 4).alias("clv_12m_prom"),
    )
    .orderBy("grupo_edad", "canal_dominante")
)
df_val.show(truncate=False)

<div style="background-color:#002147; color:white; padding:10px; border-radius:5px; font-size:18px"> 6. Visualización sobre tabla agregada y guardado en Gold

In [ ]:
pdf_seg = resumen_seg.toPandas()
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(pdf_seg["segmento_clv"], pdf_seg["clv_12m_prom"], color="#002147")
ax.set_title("CLV Proyectado 12 Meses Promedio por Segmento")
ax.set_ylabel("CLV Proyectado (unidades H&M)")
plt.tight_layout()
plt.show()

df_clv.write.mode("overwrite").parquet(f"{GOLD}/hym_clv")
print(f"Modelo guardado en {GOLD}/hym_clv | filas={spark.read.parquet(f'{GOLD}/hym_clv').count():,}")

In [ ]:
spark.stop()